# StyleTTS 2 Demo (Basque)

Inference notebook for the Basque multispeaker TTS model trained with StyleTTS2.
Uses the Basque phonemizer (`eu_phonemizer_v2`) and the WavLM-based checkpoint.

## 1. Setup & Imports

In [1]:
import torch
torch.manual_seed(0)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

import random
random.seed(0)

import numpy as np
np.random.seed(0)

In [2]:
%cd ..

/scratch/anderarrigandiaga/StyleTTS2_basque


In [3]:
import time
import yaml
import numpy as np
import torch
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as TT
import soundfile as sf
import IPython.display as ipd

from models import *
from utils import *
from text_utils import TextCleaner

%matplotlib inline

## 2. Configuration & Device Setup

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# ── paths ──────────────────────────────────────────────────────────────────────
CONFIG_PATH     = 'Configs/config_basque_multispeaker_phoneme_wavlm_800_2nd_normal.yml'
CHECKPOINT_PATH = 'Models/Basque_Multispeaker_Phoneme_wavlm_normal/epoch_2nd_00030.pth'

# Reference audio files (located in Demo/)
REFERENCE_WAVS = {
    'antton': 'Demo/ref_antton.wav',
    'maider': 'Demo/ref_maider.wav',
}

# ── config ─────────────────────────────────────────────────────────────────────
config = yaml.safe_load(open(CONFIG_PATH))
model_params = recursive_munch(config['model_params'])

# Mel-spectrogram preprocessing parameters (read from config)
_pp = config.get('preprocess_params', {})
_sp = _pp.get('spect_params', {})
MEL_PARAMS = dict(
    n_mels      = model_params.n_mels if hasattr(model_params, 'n_mels') else 80,
    n_fft       = _sp.get('n_fft',       2048),
    win_length  = _sp.get('win_length',  1200),
    hop_length  = _sp.get('hop_length',  300),
    mean        = _pp.get('mean', -4),
    std         = _pp.get('std',   4),
    sr          = _pp.get('sr',  24000),
)
print('Mel params:', MEL_PARAMS)

Using device: cpu
Mel params: {'n_mels': 80, 'n_fft': 2048, 'win_length': 1200, 'hop_length': 300, 'mean': -4, 'std': 4, 'sr': 24000}


## 3. Load Models

In [6]:
# ── ASR (text aligner) ─────────────────────────────────────────────────────────
ASR_config = config.get('ASR_config', False)
ASR_path   = config.get('ASR_path',   False)
ASR_module = config.get('ASR_module', None)   # e.g. "ASR_basque"
text_aligner = load_ASR_models(ASR_path, ASR_config, ASR_module)

# ── F0 / pitch extractor ───────────────────────────────────────────────────────
F0_path = config.get('F0_path', False)
pitch_extractor = load_F0_models(F0_path)

# ── PLBERT ─────────────────────────────────────────────────────────────────────
BERT_path = config.get('PLBERT_dir', False)
if 'phoneme' in BERT_path:
    from Utils.PLBERT_phoneme.util import load_plbert
elif 'subword' in BERT_path:
    from Utils.PLBERT_subword.util import load_plbert
elif 'naive' in BERT_path:
    from Utils.PLBERT_naive.util import load_plbert
else:
    from Utils.PLBERT_all_languages.util import load_plbert
plbert = load_plbert(BERT_path)

# ── Build model ────────────────────────────────────────────────────────────────
model = build_model(model_params, text_aligner, pitch_extractor, plbert)
_ = [model[key].eval() for key in model]
_ = [model[key].to(device) for key in model]
print('Model built.')

✅ Using ASR module: ASR_basque
ASR module file: /scratch/anderarrigandiaga/StyleTTS2_basque/Utils/ASR_basque/models.py


/scratch/anderarrigandiaga/StyleTTS2_basque/models.py:588: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  params = torch.load(path, map_location='cpu')['net']


Model built.


In [7]:
# ── Load checkpoint ────────────────────────────────────────────────────────────
params_whole = torch.load(CHECKPOINT_PATH, map_location='cpu')
params = params_whole.get('net', params_whole)

for key in model:
    if key in params:
        print(f'{key} loaded')
        try:
            model[key].load_state_dict(params[key])
        except Exception:
            from collections import OrderedDict
            new_state = OrderedDict()
            for k, v in params[key].items():
                name = k[7:] if k.startswith('module.') else k
                new_state[name] = v
            model[key].load_state_dict(new_state, strict=False)

_ = [model[key].eval() for key in model]
print('Checkpoint loaded.')

# ── Diffusion sampler ──────────────────────────────────────────────────────────
from Modules.diffusion.sampler import DiffusionSampler, ADPM2Sampler, KarrasSchedule

sampler = DiffusionSampler(
    model.diffusion.diffusion,
    sampler=ADPM2Sampler(),
    sigma_schedule=KarrasSchedule(sigma_min=0.0001, sigma_max=3.0, rho=9.0),
    clamp=False,
)
print('Sampler ready.')

/tmp/ipykernel_1286199/370737808.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  params_whole = torch.load(CHECKPOINT_PATH, map_location='cpu')


bert loaded
bert_encoder loaded
predictor loaded
decoder loaded
text_encoder loaded
predictor_encoder loaded
style_encoder loaded
diffusion loaded
text_aligner loaded
pitch_extractor loaded
mpd loaded
msd loaded
wd loaded
Checkpoint loaded.
Sampler ready.


## 4. Audio Preprocessing Utilities

In [8]:
def preprocess(wave,
               n_mels=MEL_PARAMS['n_mels'], n_fft=MEL_PARAMS['n_fft'],
               win_length=MEL_PARAMS['win_length'], hop_length=MEL_PARAMS['hop_length'],
               mean=MEL_PARAMS['mean'], std=MEL_PARAMS['std']):
    """Convert a 1-D numpy waveform to a log-mel spectrogram tensor."""
    to_mel = TT.MelSpectrogram(n_mels=n_mels, n_fft=n_fft,
                                win_length=win_length, hop_length=hop_length)
    wave_tensor = torch.from_numpy(wave).float()
    mel_tensor  = to_mel(wave_tensor)
    mel_tensor  = (torch.log(1e-5 + mel_tensor.unsqueeze(0)) - mean) / std
    return mel_tensor


def length_to_mask(lengths):
    """Build boolean padding mask from length tensor."""
    mask = torch.arange(lengths.max()).unsqueeze(0).expand(lengths.shape[0], -1).type_as(lengths)
    mask = torch.gt(mask + 1, lengths.unsqueeze(1))
    return mask


def load_and_trim(path, target_sr=24000, top_db=30):
    """Load a WAV, convert to mono at 24 kHz and trim leading/trailing silence."""
    wav, orig_sr = torchaudio.load(path)
    wav = wav.mean(0)  # stereo → mono
    if orig_sr != target_sr:
        wav = TT.Resample(orig_sr, target_sr)(wav)

    # RMS-based silence trim (equivalent to librosa.effects.trim)
    frame_length, hop_length = 2048, 512
    if len(wav) >= frame_length:
        frames = wav.unfold(0, frame_length, hop_length)
        rms    = frames.pow(2).mean(-1).sqrt()
        ref    = rms.max()
        if ref > 0:
            threshold  = ref * (10 ** (-top_db / 20.0))
            nonsilent  = (rms > threshold).nonzero(as_tuple=False).squeeze(-1)
            if nonsilent.numel() > 0:
                start = nonsilent[0].item() * hop_length
                end   = min((nonsilent[-1].item() + 1) * hop_length + frame_length, len(wav))
                wav   = wav[start:end]
    return wav.numpy()

print('Preprocessing utilities defined.')

Preprocessing utilities defined.


## 5. Style Embedding Computation

In [9]:
def compute_style(path):
    """Compute a style embedding from a reference WAV file."""
    audio      = load_and_trim(path, target_sr=MEL_PARAMS['sr'])
    mel_tensor = preprocess(audio).to(device)

    # Conv layers in the style encoder need at least 16 time frames
    min_frames = 16
    if mel_tensor.size(-1) < min_frames:
        mel_tensor = F.pad(mel_tensor, (0, min_frames - mel_tensor.size(-1)))

    with torch.no_grad():
        ref_s = model.style_encoder(mel_tensor.unsqueeze(1))
        ref_p = model.predictor_encoder(mel_tensor.unsqueeze(1))
    return torch.cat([ref_s, ref_p], dim=1)

print('compute_style defined.')

compute_style defined.


## 6. Basque Phonemizer Setup

In [10]:
import os
from Phonemizer.eu_phonemizer_v2 import Phonemizer

# Use absolute paths: _validate_paths() checks the path directly (cwd-relative),
# while _build_*_command() prepends _get_file_path() (Phonemizer/ dir).
# Absolute paths make both code-paths resolve correctly.
phon = Phonemizer(
    language='eu',
    symbol='ipa',
    path_modulo1y2=os.path.abspath('Phonemizer/modulo1y2/modulo1y2'),
    path_dicts=os.path.abspath('Phonemizer/dict'),
)

textcleaner = TextCleaner()

# Quick sanity check
_test_text  = 'Kaixo mundua!'
_test_norm  = phon.normalize(_test_text)
_test_phon  = phon.getPhonemes(_test_norm, use_single_char=True)
print(f'Input:    {_test_text}')
print(f'Norm:     {_test_norm}')
print(f'Phonemes: {_test_phon}')
print('Phonemizer ready.')

177
Input:    Kaixo mundua!
Norm:     Kaixo mundua!
Phonemes: kajʃO | mundUa | !
Phonemizer ready.


## 7. Inference Function

In [11]:
def _phonemize(text):
    """Normalize + phonemize a Basque text string → phoneme string."""
    # Ensure the sentence ends with punctuation so the duration predictor
    # doesn't produce unbounded output on fragment sentences.
    text = text.strip()
    if text and text[-1] not in '.!?;:':
        text = text + '.'

    phonemes = None
    for cand in [text.replace('\u2026', '...'), text]:
        try:
            norm     = phon.normalize(cand)
            phonemes = phon.getPhonemes(norm, use_single_char=True)
        except UnicodeDecodeError:
            try:
                safe     = cand.encode('ISO-8859-15', errors='replace').decode('ISO-8859-15')
                norm     = phon.normalize(safe)
                phonemes = phon.getPhonemes(norm, use_single_char=True)
            except Exception:
                phonemes = None
        except Exception as e:
            print(f'Warning: phonemizer error: {e}')
            phonemes = None
        if phonemes:
            break

    if phonemes:
        phonemes = phonemes.replace('\n', ' ').replace(' | ', ' ').replace('|', ' ')
        phonemes = ' '.join(phonemes.split())
    else:
        phonemes = ' '.join(text.split())  # graceful fallback
    return phonemes


def inference(text, ref_s, alpha=0.3, beta=0.7, diffusion_steps=5, embedding_scale=1.0):
    """Synthesize Basque speech from raw text.

    Args:
        text            : Input Basque text.
        ref_s           : Style embedding from compute_style().
        alpha           : Timbre mixing (0 = fully reference, 1 = fully sampled).
        beta            : Prosody mixing (0 = fully reference, 1 = fully sampled).
        diffusion_steps : Number of diffusion sampling steps.
        embedding_scale : Classifier-free guidance scale (>1 → more expressive).
    Returns:
        numpy array of the synthesized waveform at 24 000 Hz.
    """
    phonemes = _phonemize(text)
    tokens   = textcleaner(phonemes)
    tokens.insert(0, 0)
    tokens = torch.LongTensor(tokens).to(device).unsqueeze(0)

    with torch.no_grad():
        input_lengths = torch.LongTensor([tokens.shape[-1]]).to(device)
        text_mask     = length_to_mask(input_lengths).to(device)

        t_en     = model.text_encoder(tokens, input_lengths, text_mask)
        bert_dur = model.bert(tokens, attention_mask=(~text_mask).int())
        d_en     = model.bert_encoder(bert_dur).transpose(-1, -2)

        s_pred = sampler(
            noise           = torch.randn((1, 256)).unsqueeze(1).to(device),
            embedding       = bert_dur,
            embedding_scale = embedding_scale,
            features        = ref_s,
            num_steps       = diffusion_steps,
        ).squeeze(1)

        s   = s_pred[:, 128:]
        ref = s_pred[:, :128]
        ref = alpha * ref + (1 - alpha) * ref_s[:, :128]
        s   = beta  * s   + (1 - beta)  * ref_s[:, 128:]

        d        = model.predictor.text_encoder(d_en, s, input_lengths, text_mask)
        x, _     = model.predictor.lstm(d)
        duration = model.predictor.duration_proj(x)
        duration = torch.sigmoid(duration).sum(axis=-1)
        pred_dur = torch.round(duration.squeeze()).clamp(min=1)
        if pred_dur.dim() == 0:
            pred_dur = pred_dur.unsqueeze(0)

        seq_len  = int(input_lengths.item())
        total_dur = int(pred_dur.sum().item())
        pred_aln_trg = torch.zeros(seq_len, total_dur)
        c_frame = 0
        for i in range(seq_len):
            dur_i = int(pred_dur[i].item())
            pred_aln_trg[i, c_frame:c_frame + dur_i] = 1
            c_frame += dur_i

        en  = (d.transpose(-1, -2) @ pred_aln_trg.unsqueeze(0).to(device))
        asr = (t_en @ pred_aln_trg.unsqueeze(0).to(device))
        if model_params.decoder.type == 'hifigan':
            for _t in [en, asr]:
                pass  # handled below
            en_new  = torch.zeros_like(en);  en_new[:, :, 0] = en[:, :, 0];  en_new[:, :, 1:] = en[:, :, 0:-1];  en  = en_new
            asr_new = torch.zeros_like(asr); asr_new[:, :, 0] = asr[:, :, 0]; asr_new[:, :, 1:] = asr[:, :, 0:-1]; asr = asr_new

        F0_pred, N_pred = model.predictor.F0Ntrain(en, s)
        out = model.decoder(asr, F0_pred, N_pred, ref.squeeze().unsqueeze(0))

    return out.squeeze().cpu().numpy()[..., :-50]  # trim artefact pulse at end

print('inference() defined.')

inference() defined.


## 8. Basic Synthesis

Synthesize a Basque sentence with each reference speaker.
`alpha` controls timbre mixing, `beta` controls prosody mixing (both relative to the diffusion-sampled style vs. the reference style).

In [12]:
# ── Configure these two lines ──────────────────────────────────────────────────
text    = 'Arratsalde on Cesar, hiru zerbeza mesedez?'
speaker = 'antton'   # 'antton' or 'maider'
# ──────────────────────────────────────────────────────────────────────────────

ref_path = REFERENCE_WAVS[speaker]
ref_s    = compute_style(ref_path)

t0  = time.time()
wav = inference(text, ref_s, alpha=0.3, beta=0.7, diffusion_steps=5, embedding_scale=1.0)
rtf = (time.time() - t0) / (len(wav) / 24000)

print(f'Speaker: {speaker}')
print(f'Text:    {text}')
print(f'RTF:     {rtf:.3f}')
print('Synthesized:')
display(ipd.Audio(wav, rate=24000, normalize=False))

: 

### With more diffusion steps (more diverse)

Higher step count → slower but more expressive/diverse samples.

In [ ]:
torch.cuda.empty_cache()

text  = 'Benetan uste duzu jokabide horrekin urrunera helduko zarela?'
ref_s = compute_style(REFERENCE_WAVS['antton'])

for steps in [5, 10]:
    torch.cuda.empty_cache()
    t0  = time.time()
    wav = inference(text, ref_s, alpha=0.3, beta=0.7, diffusion_steps=steps, embedding_scale=1.0)
    rtf = (time.time() - t0) / (len(wav) / 24000)
    print(f'diffusion_steps={steps}  RTF={rtf:.3f}')
    display(ipd.Audio(wav, rate=24000, normalize=False))

: 

### Effect of `embedding_scale` (classifier-free guidance)

Higher `embedding_scale` makes the style more conditioned on the text, producing more expressive speech.

In [ ]:
text  = 'Izugarri gustatzen zait!'
ref_s = compute_style(REFERENCE_WAVS['antton'])

for scale in [1.0, 1.5, 2.0]:
    wav = inference(text, ref_s, alpha=0.3, beta=0.7, diffusion_steps=10, embedding_scale=scale)
    print(f'embedding_scale={scale}')
    display(ipd.Audio(wav, rate=24000, normalize=False))

## (Optional) Save outputs to disk

In [ ]:
import os

OUT_DIR = 'output/basque_demo'
os.makedirs(OUT_DIR, exist_ok=True)

save_texts = {
    'TST_00005': 'Tailandiara bidaiatzea gustatuko litzaidake, Bangkok ezagutu nahiko nuke.',
    'TST_00012': 'Arratsalde on, zerbeza pare bat mesedez?',
    'TST_00022': 'Sei zerri zahar zuhur zor.',
}

for speaker, ref_path in REFERENCE_WAVS.items():
    ref_s = compute_style(ref_path)
    for key, text in save_texts.items():
        wav      = inference(text, ref_s, alpha=0.3, beta=0.7, diffusion_steps=5)
        out_path = os.path.join(OUT_DIR, f'{speaker}_{key}_synth.wav')
        sf.write(out_path, wav, 24000)
        print(f'Saved: {out_path}')

print('Done.')